In [ ]:
# Cell 1: Secrets
import os
from google.colab import userdata






In [ ]:
# Cell 2: Setup
!git clone https://github.com/sk3feel/hidden-data-reproduction-multimodal.git /content/course_work2026 2>/dev/null || (cd /content/course_work2026 && git pull)
%cd /content/course_work2026
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/course_work2026')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print({'drive_root': str(DRIVE_ROOT)})
!pip install -q -r requirements.txt
!pip install -q accelerate Pillow pandas tqdm einops timm comet_ml huggingface_hub
!pip install -q peft bitsandbytes trl qwen-vl-utils
!pip uninstall -y transformers tokenizers
!pip install -q tokenizers==0.21.0
!pip install -q transformers==4.49.0 --force-reinstall --no-deps
!python src/download_from_hf.py --repo-id sk3feel/docvqa-privacy-data
assert Path('artifacts/finetuning_generative/train_florence2.jsonl').exists()
assert Path('artifacts/docqa_recovery/benchmark_train/images/original').exists()


In [ ]:
# Cell 3: Imports and constants
import json
import os
import re
from collections import defaultdict
from pathlib import Path

import comet_ml
import pandas as pd
import torch
from PIL import Image, ImageFile
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoProcessor
from huggingface_hub import HfApi, login
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import transformers
print(f'transformers version: {transformers.__version__}')
if not transformers.__version__.startswith('4.4'):
    print(f'WARNING: expected 4.4x.x, got {transformers.__version__}. Florence-2 may fail.')

ImageFile.LOAD_TRUNCATED_IMAGES = True

MODEL_NAME = 'microsoft/Florence-2-base'
EPOCHS = 10
BATCH_SIZE = 2
LR = 1e-5
CHECKPOINT_EPOCHS = [3, 5, 10]
HF_MODEL_REPO = 'sk3feel/docvqa-privacy-checkpoints'
TRAIN_JSONL = Path('artifacts/finetuning_generative/train_florence2.jsonl')
VALIDATION_JSONL = Path('artifacts/finetuning_generative/validation_florence2.jsonl')
DRIVE_ARTIFACTS_ROOT = DRIVE_ROOT / 'artifacts_v2_controlled'
DRIVE_ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
FINETUNING_DIR = DRIVE_ARTIFACTS_ROOT / 'finetuning_generative'
FINETUNING_DIR.mkdir(parents=True, exist_ok=True)
COMMON_TRAIN_IDS_PATH = FINETUNING_DIR / 'common_train_200_ids.csv'
COMMON_UNSEEN_IDS_PATH = FINETUNING_DIR / 'common_unseen_200_ids.csv'

experiment = comet_ml.Experiment(api_key=os.environ['COMET_API_KEY'], workspace='scfeel', project_name='qwen3-1')
experiment.set_name('florence2-finetune-v2')
experiment.log_parameters({'model': MODEL_NAME, 'epochs': EPOCHS, 'lr': LR, 'batch_size': BATCH_SIZE, 'variant': 'v2_controlled', 'train_size': 200})
login(token=os.environ['HF_TOKEN'])
api = HfApi()
api.create_repo(repo_id=HF_MODEL_REPO, repo_type='model', private=True, exist_ok=True)


In [ ]:
# Cell 4: Load data
QUESTION_RE = re.compile(r'<Question>(.*?)</Question>', flags=re.IGNORECASE | re.DOTALL)

def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def extract_question(row: dict) -> str:
    if row.get('question'):
        return str(row['question']).strip()
    if row.get('prompt'):
        return str(row['prompt']).strip()
    task_prompt = str(row.get('task_prompt', '')).strip()
    match = QUESTION_RE.search(task_prompt)
    if match:
        return match.group(1).strip()
    return task_prompt.replace('<VQA>', '').strip()

def resolve_image_path(raw_path: str, split: str) -> Path:
    path = Path(raw_path)
    if path.exists():
        return path
    image_name = raw_path.replace('\\', '/').split('/')[-1]
    split_dir = 'benchmark_train' if split == 'train' else 'benchmark'
    candidate = Path('artifacts') / 'docqa_recovery' / split_dir / 'images' / 'original' / image_name
    if candidate.exists():
        return candidate
    raise FileNotFoundError(f'Image not found: {raw_path}')

def prepare_records(rows: list[dict], default_split: str) -> list[dict]:
    records = []
    for idx, row in enumerate(rows):
        split = str(row.get('split', default_split))
        records.append({'example_id': str(row.get('example_id', row.get('question_id', idx))), 'image_path': str(resolve_image_path(str(row['image_path']), split)), 'question': extract_question(row), 'answer': str(row['answer']).strip(), 'split': split})
    return records

def enrich_with_field_types(records, manifest_path):
    manifest = load_jsonl(manifest_path)
    type_map = {}
    for row in manifest:
        eid = str(row.get('example_id', row.get('question_id', '')))
        type_map[eid] = row.get('coarse_field_type', 'OTHER')
    for rec in records:
        rec['coarse_field_type'] = type_map.get(rec['example_id'], 'OTHER')
    return records

def stratified_sample(records, n, type_key="coarse_field_type"):
    by_type = defaultdict(list)
    for rec in records:
        by_type[rec[type_key]].append(rec)
    types = sorted(by_type.keys())
    if not types:
        return []
    base = n // len(types)
    remainder = n % len(types)
    target_counts = {ft: base + (1 if idx < remainder else 0) for idx, ft in enumerate(types)}
    result = []
    used = set()
    for ft in types:
        selected = by_type[ft][:target_counts[ft]]
        result.extend(selected)
        used.update(rec["example_id"] for rec in selected)
    if len(result) < n:
        for rec in records:
            if len(result) >= n:
                break
            if rec["example_id"] not in used:
                result.append(rec)
                used.add(rec["example_id"])
    return result[:n]

all_train_records = prepare_records(load_jsonl(TRAIN_JSONL), 'train')
all_train_records = enrich_with_field_types(all_train_records, Path('artifacts/docqa_recovery/benchmark_train/manifest.jsonl'))
validation_records = prepare_records(load_jsonl(VALIDATION_JSONL), 'validation')
validation_records = enrich_with_field_types(validation_records, Path('artifacts/docqa_recovery/benchmark/manifest.jsonl'))

if COMMON_TRAIN_IDS_PATH.exists():
    common_train_ids = pd.read_csv(COMMON_TRAIN_IDS_PATH)['example_id'].astype(str).tolist()
else:
    common_train_ids = [r['example_id'] for r in stratified_sample(all_train_records, 200)]
    pd.DataFrame({'example_id': common_train_ids}).to_csv(COMMON_TRAIN_IDS_PATH, index=False)

if COMMON_UNSEEN_IDS_PATH.exists():
    common_unseen_ids = pd.read_csv(COMMON_UNSEEN_IDS_PATH)['example_id'].astype(str).tolist()
else:
    common_unseen_ids = [r['example_id'] for r in stratified_sample(validation_records, 200)]
    pd.DataFrame({'example_id': common_unseen_ids}).to_csv(COMMON_UNSEEN_IDS_PATH, index=False)

train_map = {r['example_id']: r for r in all_train_records}
val_map = {r['example_id']: r for r in validation_records}
train_records = [train_map[eid] for eid in common_train_ids if eid in train_map]
validation_records = [val_map[eid] for eid in common_unseen_ids if eid in val_map]

print({'train_records': len(train_records), 'validation_records': len(validation_records)})
pd.DataFrame(train_records[:3])[['example_id','question','answer','coarse_field_type']]


In [ ]:
# Cell 5: Dataset
class Florence2Dataset(Dataset):
    def __init__(self, records, processor):
        self.records = records
        self.processor = processor

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        image = Image.open(rec["image_path"]).convert("RGB")
        prompt = f"<VQA>{rec['question']}"
        inputs = self.processor(text=prompt, images=image, return_tensors="pt", padding=True, truncation=True)
        labels = self.processor.tokenizer(text=rec["answer"], return_tensors="pt", padding=True, truncation=True).input_ids
        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "pixel_values": inputs["pixel_values"].squeeze(),
            "labels": labels.squeeze(),
        }


def make_collate_fn(processor):
    pad_token_id = processor.tokenizer.pad_token_id or 0

    def collate_fn(batch):
        input_ids = torch.nn.utils.rnn.pad_sequence(
            [item["input_ids"] for item in batch],
            batch_first=True,
            padding_value=pad_token_id,
        )
        labels = torch.nn.utils.rnn.pad_sequence(
            [item["labels"] for item in batch],
            batch_first=True,
            padding_value=-100,
        )
        pixel_values = torch.stack([item["pixel_values"] for item in batch])
        return {
            "input_ids": input_ids,
            "pixel_values": pixel_values,
            "labels": labels,
        }

    return collate_fn


In [ ]:
# Cell 6: Training loop
device = "cuda" if torch.cuda.is_available() else "cpu"
if device != "cuda":
    raise RuntimeError("GPU runtime is required")

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True).to("cuda")
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
dataset = Florence2Dataset(train_records, processor)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=make_collate_fn(processor))
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
train_loss_history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss, valid_batches = 0, 0
    for batch in tqdm(dataloader, desc=f"Epoch {epoch}/{EPOCHS}"):
        batch = {k: v.to("cuda") for k, v in batch.items()}
        outputs = model(input_ids=batch["input_ids"], pixel_values=batch["pixel_values"], labels=batch["labels"])
        loss = outputs.loss
        if not loss.isfinite():
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()
        epoch_loss += loss.item()
        valid_batches += 1
    avg_loss = epoch_loss / max(valid_batches, 1)
    train_loss_history.append({"epoch": epoch, "train_loss": avg_loss})
    print(f"Epoch {epoch}, loss: {avg_loss:.4f}")
    experiment.log_metric("train_loss", avg_loss, step=epoch, epoch=epoch)

    if epoch in CHECKPOINT_EPOCHS:
        ckpt = f"checkpoints/florence2_v2/epoch_{epoch}"
        os.makedirs(ckpt, exist_ok=True)
        model.save_pretrained(ckpt)
        processor.save_pretrained(ckpt)
        api.upload_folder(folder_path=ckpt, repo_id=HF_MODEL_REPO, path_in_repo=f"florence2_v2/epoch_{epoch}", repo_type="model")

        model.eval()
        correct = 0
        for rec in train_records[:20]:
            img = Image.open(rec["image_path"]).convert("RGB")
            inp = processor(text=f"<VQA>{rec['question']}", images=img, return_tensors="pt").to("cuda")
            generate_inputs = {"input_ids": inp["input_ids"], "pixel_values": inp["pixel_values"]}
            with torch.no_grad():
                gen = model.generate(**generate_inputs, max_new_tokens=50)
            pred = processor.batch_decode(gen, skip_special_tokens=True)[0].strip()
            if pred.lower() == rec["answer"].lower():
                correct += 1
        experiment.log_metric("sanity_em", correct / 20, step=epoch, epoch=epoch)
        model.train()

train_loss_df = pd.DataFrame(train_loss_history)
train_loss_path = FINETUNING_DIR / 'losses_run1.csv'
train_loss_df.to_csv(train_loss_path, index=False)
experiment.log_asset(str(train_loss_path), file_name=train_loss_path.name)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(train_loss_df["epoch"], train_loss_df["train_loss"], marker="o")
ax.set_title("Florence-2 v2 train loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Train loss")
ax.grid(alpha=0.3)
plt.tight_layout()
experiment.log_figure("train_loss_curve", fig)
plt.close(fig)


In [ ]:
# Cell 7: Final sanity
final_ckpt = Path("checkpoints/florence2_v2/epoch_10")
assert final_ckpt.exists(), f"Missing checkpoint: {final_ckpt}"

final_model = AutoModelForCausalLM.from_pretrained(final_ckpt, trust_remote_code=True).to("cuda")
final_processor = AutoProcessor.from_pretrained(final_ckpt, trust_remote_code=True)
final_model.eval()


def normalize_text(text: str) -> str:
    return " ".join(str(text).strip().lower().split())


def run_final_sanity(records: list[dict], split_name: str) -> tuple[list[dict], float]:
    rows = []
    correct = 0
    for rec in records[:100]:
        img = Image.open(rec["image_path"]).convert("RGB")
        inp = final_processor(text=f"<VQA>{rec['question']}", images=img, return_tensors="pt").to("cuda")
        generate_inputs = {
            "input_ids": inp["input_ids"],
            "pixel_values": inp["pixel_values"],
        }
        with torch.no_grad():
            gen = final_model.generate(**generate_inputs, max_new_tokens=50)
        pred = final_processor.batch_decode(gen, skip_special_tokens=True)[0].strip()
        is_match = normalize_text(pred) == normalize_text(rec["answer"])
        correct += int(is_match)
        rows.append(
            {
                "split": split_name,
                "question": rec["question"],
                "gold": rec["answer"],
                "prediction": pred,
                "exact_match": is_match,
            }
        )
    return rows, correct / max(len(rows), 1)



seen_rows, seen_em = run_final_sanity(train_records, "seen")
unseen_rows, unseen_em = run_final_sanity(validation_records, "unseen")

print({"seen_em": seen_em, "unseen_em": unseen_em})

sanity_final_df = pd.DataFrame(seen_rows + unseen_rows)
sanity_path = FINETUNING_DIR / 'florence2_v2_sanity_final.csv'
sanity_final_df.to_csv(sanity_path, index=False)
experiment.log_asset(str(sanity_path), file_name=sanity_path.name)
experiment.log_metric("sanity_final_seen_em", seen_em, step=10, epoch=10)
experiment.log_metric("sanity_final_unseen_em", unseen_em, step=10, epoch=10)
experiment.log_table("sanity_final.csv", sanity_final_df)
display(sanity_final_df.head(10))


In [ ]:
# Cell 8: Finish
experiment.end()


In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_TOKEN"])
files = api.list_repo_files("sk3feel/docvqa-privacy-checkpoints", repo_type="model")

florence_files = sorted([f for f in files if f.startswith("florence2/")])
print(*florence_files, sep="\n")
